# AI4SCIENCE Hippocampus Demo

This is a demo of the ai4science(hippocampus) API through the Python client. The following features are demonstrated: 
- Dataset provisioning
- Running OpenML, Hugging Face jobs
- Explicit resource definition
- Working with BYO artifacts
- Automatic tier routing across compute clusters.

## Coming soon

Not yet available through the API -- in progress or planned:

- **Agentic workflows** -- Agentic driven workflows with hippocampus as the slurm orchestrator. Think agents making agents that are spawned and run on snellius via hippocampus
- **Ray-based multi-node distributed jobs** -- running a job across more
  than one node with Ray for coordination. Not yet available; single-node
  jobs only for now.
- **More Monitoring** -- beyond streaming logs to mlflows and other job monitoring tools

Set `AI4SCIENCE_USER` and `AI4SCIENCE_TOKEN` below before running.

In [2]:
import os

BASE_URL = "https://ai4science.dev.sdp.surf.nl"
USER = os.environ.get("AI4SCIENCE_USER", "juliusa")
TOKEN = os.environ.get("AI4SCIENCE_TOKEN", "")
assert USER, "Set AI4SCIENCE_USER (your Snellius username) before running this notebook."
assert TOKEN, "Set AI4SCIENCE_TOKEN (a fresh SLURM JWT, e.g. via scontrol token) before running this notebook."

print(f"Ready. base_url={BASE_URL}, user={USER}")

Ready. base_url=https://ai4science.dev.sdp.surf.nl, user=juliusa


## Client setup

One client instance, reused across every example below.

In [3]:
from ai4science_client import Ai4ScienceClient

client = Ai4ScienceClient(base_url=BASE_URL, user=USER, token=TOKEN)
print("Client ready.")

Client ready.


## FEATURE: Data provisioning

Stage a public dataset onto Snellius via the raw API (`POST /datasets/add`),
then poll `POST /datasets/status` until it's ready. 

Ideally there is an approval/guardrail step between request and staging.

This has been skipped here for demo.

Also perhaps we could have a silent auto garbage collector on unused datasets.

In [4]:
import time

import requests

DATASET_ID, PLATFORM = "GPT-NL/GPT-NL_Public_Corpus", "Hugging Face"
clean_name = DATASET_ID.split("/")[-1]

### PROVISION REQUEST ######
resp = requests.post(f"{BASE_URL}/datasets/add", json={
    "dataset_id": DATASET_ID, "platform": PLATFORM, "user": USER, "token": TOKEN,
})
resp.raise_for_status()
submitted = resp.json()
print("Provisioning job submitted:\n", submitted)

##### STATUS REQUEST #######
state = "PENDING"
while state not in ("READY", "FAILED"):
    time.sleep(10)
    registry = requests.post(f"{BASE_URL}/datasets/status", json={"datasets": [clean_name]}).json()
    state = registry.get(clean_name, {}).get("status", "unknown").upper()
    print("\n\nFinal provisioning status:", state)

HTTPError: 502 Server Error: Bad Gateway for url: https://ai4science.dev.sdp.surf.nl/datasets/add

## FEATURE: OpenML job via the `@job` decorator

A real OpenML workflow -- fetch a task, train a model, report the score --
running on Snellius. 

`@job(...)` turns a plain Python function into a
remote job with near-zero snellius env config at the call site.

In [5]:
from ai4science_client import job


@job(base_url=BASE_URL, user=USER, token=TOKEN, dependencies=["openml", "scikit-learn"], stream=True) # only line to run job everything else is user code
def run_openml_task(task_id: int) -> dict:
    import time
    import openml
    from sklearn.ensemble import RandomForestClassifier

    def get_task_with_retries(task_id, attempts=3, delay=10):
        for attempt in range(1, attempts + 1):
            try:
                return openml.tasks.get_task(task_id)
            except Exception as e:
                print(f"openml.org fetch failed (attempt {attempt}/{attempts}): {e}")
                if attempt == attempts:
                    raise
                time.sleep(delay)

    task = get_task_with_retries(task_id)
    X, y = task.get_X_and_y()
    train_idx, test_idx = task.get_train_test_split_indices()

    clf = RandomForestClassifier(n_estimators=100)
    clf.fit(X[train_idx], y[train_idx])
    accuracy = clf.score(X[test_idx], y[test_idx])

    return {"task_id": task_id, "accuracy": accuracy}


result = run_openml_task(31)
print(result)

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): openml scikit-learn ===
=== [3/3] Running Python Script: /projects/2/managed_datasets/containers/ai4science/jobs/api_job_6550b898.py ===
/projects/2/managed_datasets/containers/ai4science/jobs/api_job_6550b898.py:17: FutureWarning: Support for `dataset_format='array'` will be removed in 0.15,start using `dataset_format='dataframe' to ensure your code will continue to work. You can use the dataframe's `to_numpy` function to continue using numpy arrays.
  X, y = task.get_X_and_y()
/opt/conda/lib/python3.11/site-packages/openml/tasks/task.py:334: FutureWarning: Support for `dataset_format='array'` will be removed in 0.15,start using `dataset_format='dataframe' to ensure your code will continue to work. You can use the dataframe's `to_numpy` function to continue using numpy arrays.
  X, y, _, _ = dataset.get_data(
[result received: {'task_id': 31, 'accuracy': 0.81}]
INFO:    Cleanup error: wh

## FEATURE: Hugging Face job with explicit resources + a local artifact

Run a Hugging Face model with explicit resource requirements
(`SlurmResourceConfig`).

And pass in a local file as an artifact -- uploaded
automatically, available inside the job as a normal function argument, no
extra code needed to fetch it.

In [6]:
with open("sample_texts.txt", "w") as f:
    f.write("Snellius makes large-scale AI research so much easier.\n")
    f.write("I really dislike waiting for slow pip installs.\n")
    f.write("This dataset looks promising for our next experiment.\n")

In [7]:
from ai4science_client.schemas import SlurmResourceConfig

resources = SlurmResourceConfig(
    partition="gpu_h100",
    cpus_per_task=8,
    memory_mb=32000,
    time_limit_minutes=20,
    tres_per_node="gres:gpu:1",
)


def classify_with_local_texts(data_path: str) -> dict:
    import torch
    from transformers import pipeline

    with open(data_path) as f:
        texts = [line.strip() for line in f if line.strip()]

    device = 0 if torch.cuda.is_available() else -1
    classifier = pipeline("sentiment-analysis", device=device)
    results = classifier(texts)

    return {"texts": texts, "results": results}


result = client.run(
    classify_with_local_texts,
    artifacts={"data_path": "./sample_texts.txt"},
    dependencies=["torch", "transformers<4.50"],
    resources=resources,
    stream=True,
)

print(result)

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): torch transformers<4.50 boto3 ===
/usr/bin/bash: line 2: 4.50: No such file or directory
=== Job Finished with exit code 0 ===
None


## 4. Automatic tier routing

Pass `tier="auto"` and the API estimates what a job actually needs (CPU,
memory, GPU) from its script and dependencies, then picks the smallest
compute cluster that can satisfy it -- searching every tier, cluster and partitions available for the best fit. 
    
Here, a GPU-needing job is checked against both `dev-slurm` (local slurm sim with no
GPU capability) and `snellius` (has GPU partitions) -- only Snellius can
satisfy it, so that's where it lands :))

You can also control how much of that decision the API makes for you:

```python
# Fully automatic -- search everything, pick the smallest fit
client.run(fn, tier="auto")

# Pin to a tier -- only search clusters within it
client.run(fn, tier="1")

# Pin to a specific cluster -- skip the tier/cluster search, still
# picks the smallest fitting partition on that one cluster
client.run(fn, tier="1", cluster="snellius")

# Pin cluster + exact partition -- no estimation or search at all,
# runs exactly where you say
client.run(
    fn,
    tier="1",
    cluster="snellius",
    resources=SlurmResourceConfig(partition="gpu_h100"),
)
```

In [8]:
def gpu_matrix_multiply(size: int = 2048) -> dict:
    import time

    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    a = torch.randn(size, size, device=device)
    b = torch.randn(size, size, device=device)

    torch.cuda.synchronize() if device == "cuda" else None
    start = time.time()
    c = a @ b
    torch.cuda.synchronize() if device == "cuda" else None
    elapsed = time.time() - start

    return {"device": device, "size": size, "elapsed_seconds": elapsed}


result = client.run(
    gpu_matrix_multiply,
    2048,
    dependencies=["torch"],
    tier="auto",
    stream=True,
)
print(result)

=== [1/3] Preparing Apptainer Container ===
=== [2/3] Installing Dependencies (Ephemeral Overlay): torch ===
=== [3/3] Running Python Script: /projects/2/managed_datasets/containers/ai4science/jobs/api_job_1c1654d7.py ===
[result received: {'device': 'cuda', 'size': 2048, 'elapsed_seconds': 0.2572488784790039}]
=== Job Finished with exit code 0 ===
{'device': 'cuda', 'size': 2048, 'elapsed_seconds': 0.2572488784790039}


## FEATURE: EESSI module-based job (raw API)

EESSI jobs use pre-built software modules instead of installing
dependencies at runtime -- a different execution model from
`/ephemeral-job`. 

The client doesn't wrap this one yet, so this is a
plain `POST /eessi-job` request, same shape as the dataset provisioning
call above.

In [9]:
resp = requests.post(
    f"{BASE_URL}/eessi-job",
    json={
        "modules": ["foss/2023a", "SciPy-bundle/2023.07-gfbf-2023a"],
        "python_script": (
            "import numpy as np\n"
            "a = np.random.rand(1000, 1000)\n"
            "print('Matrix sum:', a.sum())\n"
        ),
        "user": USER,
        "token": TOKEN,
    },
)
resp.raise_for_status()
submitted = resp.json()
print("EESSI job submitted:", submitted)

job_id = submitted["job_id"]
seen_len = 0
state = "pending"

while state not in ("completed", "failed"):
    time.sleep(10)

    log_resp = requests.get(f"{BASE_URL}/logs/{job_id}")
    if log_resp.status_code == 200 and len(log_resp.text) > seen_len:
        print(log_resp.text[seen_len:], end="")
        seen_len = len(log_resp.text)

    result_resp = requests.get(f"{BASE_URL}/results/{job_id}")
    if result_resp.status_code == 404:
        continue
    result_resp.raise_for_status()
    result = result_resp.json()
    state = result.get("status", "unknown")

print("\nFinal result:", result)

EESSI job submitted: {'job_id': 26584034, 'status': 'SUBMITTED', 'output_file': '/tmp/26584034.log', 'error_file': '/tmp/26584034.log', 'user': 'juliusa', 's3_key': None}
--- Phase 1: EESSI Init ---
Attempting to initialize EESSI version: 2023.06
EESSI Initialized successfully.
--- Phase 2: Module Load ---
Loading foss/2023a...
Loading SciPy-bundle/2023.07-gfbf-2023a...
--- Phase 3: Python ---
Using Python from: /cvmfs/software.eessi.io/versions/2023.06/software/linux/x86_64/amd/zen4/software/Python/3.11.3-GCCcore-12.3.0/bin/python3
Matrix sum: 499855.52372698893
=== Job Finished with exit code 0 ===

Final result: {'job_id': '26584034', 'status': 'completed', 'exit_code': 0, 'result': None}


## Other features available

Not demoed above, but supported by the client and API today:

- **Async submission** -- `client.submit(...)` returns immediately; poll or
  block on it later with `job.results()` / `job.wait()`.

- **Hugging Face/openml token passthrough** -- `hf_token=` on `run()`/`@job(...)`
  for gated/private HF models and datasets.

## Multi-node Ray jobs

Pass `container="ray"` to run against a live, multi-node Ray cluster instead of a single process. The server bootstraps the cluster (head + workers across the allocated nodes) before your function runs — call `ray.init(address="auto")` directly, no need to start Ray yourself.

`resources.nodes` controls cluster size (head + `nodes - 1` workers); every other `SlurmResourceConfig` field (`cpus_per_task`, `memory_mb`, `tres_per_node`, etc.) describes what's requested on **each** node, same fields as a single-node job, just applied per-node instead of once.

`dependencies=`, `artifacts=`, and `stream=` all work exactly the same as single-node jobs. `tier=`/`cluster=` (auto-tier-routing) are not supported with `container="ray"` and raise `ValueError` if combined.

In [ ]:
from ai4science_client import Ai4ScienceClient
from ai4science_client.schemas import SlurmResourceConfig

client = Ai4ScienceClient(base_url=BASE_URL, user=USER, token=TOKEN)


def train_distributed():
    import ray

    ray.init(address="auto")
    return {
        "node_count": len(ray.nodes()),
        "cluster_resources": ray.cluster_resources(),
    }


result = client.run(
    train_distributed,
    container="ray",
    resources=SlurmResourceConfig(
        partition="rome",
        nodes=2,
        cpus_per_task=8,
        memory_mb=16000,
        time_limit_minutes=15,
        tres_per_node="",  # explicit no-GPU override, cheap/fast for the demo
    ),
    stream=True,
)
print(result)

=== [1/4] Preparing Ray Container ===
=== [2/4] Starting Ray head on tcn148 (172.18.56.158:6379) ===
2026-09-11 07:40:38,096	INFO usage_lib.py:474 -- Usage stats collection is enabled by default without user confirmation because this terminal is detected to be non-interactive. To disable this, add `--disable-usage-stats` to the command that starts the cluster, or run the following command: `ray disable-usage-stats` before starting the cluster. See https://docs.ray.io/en/master/cluster/usage-stats.html for more details.
2026-09-11 07:40:38,098	INFO scripts.py:1042 -- Local node IP: 172.18.56.158
=== [3/4] Starting 1 Ray worker(s) ===
Starting worker on tcn172
Waiting for cluster to be ready...
2026-09-11 07:40:57,575	SUCC scripts.py:1081 -- --------------------
2026-09-11 07:40:57,575	SUCC scripts.py:1082 -- Ray runtime started.
2026-09-11 07:40:57,575	SUCC scripts.py:1083 -- --------------------
2026-09-11 07:40:57,575	INFO scripts.py:1085 -- Next steps
2026-09-11 07:40:57,575	INFO cli